# 案例：需求理解与事实推理

## 核心学习目标

本案例的核心知识点是基于提示词工程的思维链推理。通过在提示词中设置明确的分析任务和输出要求，引导模型按照预设推理思路理解用户需求，并生成分步骤的事实分析结果。

## 案例设计

为了突出这一机制，案例设计四类事实预调查方法，将相关信息划分为“已给出或已验证的事实”“需要查阅的事实”“需要推导的事实”和“有根据的猜测”。这种划分能够帮助模型识别已知条件、外部信息需求、待推导内容与不确定性信息。

**学习重点**：
1. 理解如何将需求理解与事实推理过程转化为具体的思维链提示词。
2. 使用 **LangChain `PromptTemplate`** 渲染完整提示词，并观察四类事实分析框架在模型输出中的体现。
3. 使用 **`ChatOpenAI`** 调用模型，验证模型能否按照预设框架生成结构化的分析结果。


## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 notebook 前，请在仓库根目录执行 `uv pip install -r Chapter-2/requirements.txt`，并将 notebook 内核设置为该 `.venv` 对应的 Python 解释器。

In [1]:
# 在 Windows 环境中将标准输出流设置为 UTF-8，避免输出中文或特殊字符时发生编码错误（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

In [2]:
from datetime import datetime
print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

当前时间: 2026-11-14 17:09:20 +0800


In [3]:
# 用户请求
task = """请帮我规划 2027 年 5 月 1 日至 5 日的旅行，
两个人，想去上海、苏州和杭州，还没确定城市游览顺序。
希望包含西湖，酒店每晚不超过 800 元，尽量不要安排夜间跨城交通。
请提供路线、酒店、天气和美食攻略。"""

## 1. 思维链提示词模板

该提示词将需求理解拆分为四类事实分析任务：
- 提取用户请求中已经明确给出的条件
- 识别需要进行外部查询的信息
- 确定需要通过计算或逻辑得出的结论
- 以及标记尚未验证的判断。
通过限定输出标题和内容范围，模型可以按照预设思维链生成结构化结果。

模板中占位符 `{task}`，用于传入用户的自然语言任务。

In [4]:
FACTS_PROMPT = """在处理用户任务之前，请先完成以下信息分析。回答应准确、完整，并明确区分已知信息、待查证信息、推导结果与不确定性判断。

用户输入任务：

{task}

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。
"""


## 2. 用 LangChain 封装 完整提示词

在本例中，`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 会生成相同的字符串。使用 `PromptTemplate` 便于后续集成 LCEL 并进行链式调试。

In [5]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())


assert full_prompt == FACTS_PROMPT.format(task=task.strip())

## 3. 展示完整的思维链提示词

下面展示编排器在第一步发送给模型的**完整用户消息**。如仅需展示部分内容，可使用 `full_prompt[:2000]` 进行截取。

In [6]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))

**字符数：** 522

In [7]:
display(Markdown("```text\n" + full_prompt + "\n```"))

```text
在处理用户任务之前，请先完成以下信息分析。回答应准确、完整，并明确区分已知信息、待查证信息、推导结果与不确定性判断。

用户输入任务：

请帮我规划 2027 年 5 月 1 日至 5 日的旅行，
两个人，想去上海、苏州和杭州，还没确定城市游览顺序。
希望包含西湖，酒店每晚不超过 800 元，尽量不要安排夜间跨城交通。
请提供路线、酒店、天气和美食攻略。

以下是预调查问题：

    1. 请列出请求中明确给出的任何具体事实或数据。可能没有任何此类信息。
    2. 请列出可能需要查阅的任何事实，以及具体可以在哪里找到这些信息。在某些情况下，请求本身会提及权威来源。
    3. 请列出可能需要推导的任何事实（例如，通过逻辑演绎、模拟或计算得出）。
    4. 请列出从记忆中回忆出的任何事实、直觉、经过充分推理的猜测等。

在回答此调查时，请记住，"事实"通常是具体的名称、日期、统计数据等。您的回答应使用以下标题：

    1. 已给出或已验证的事实
    2. 需要查阅的事实
    3. 需要推导的事实
    4. 有根据的猜测

不要在你的回复中包含其他任何标题或部分。在被要求之前，不要列出下一步行动或计划。

```

## 4.使用 LangChain `ChatOpenAI` 调用模型

如果希望实际调用 Qwen 执行上述思维链提示词，请先将仓库根目录的 `.env.example` 复制为 `.env`，并至少配置 `DASHSCOPE_API_KEY`。可通过 `DASHSCOPE_CHAT_MODEL` 指定模型，未配置时默认使用 `qwen3.7-plus`。以下代码会自动向上查找并加载 `.env`，无需在 notebook 中写入 API Key。

新版 Qwen 的 thinking 模式会显著增加响应时间。本案例只需要按既定框架整理信息，因此调用时显式关闭 thinking，并设置超时，避免长时间没有输出而被误认为内核卡死。

In [8]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

kw = dict(
    model=os.getenv("DASHSCOPE_CHAT_MODEL", "qwen3.7-plus"),
    temperature=0,
    base_url=os.getenv("DASHSCOPE_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
    api_key=os.environ["DASHSCOPE_API_KEY"],
    extra_body={"enable_thinking": False},
    timeout=60,
    max_retries=1,
)

llm = ChatOpenAI(**kw)
print(f"正在调用 {kw['model']}（已关闭 thinking），请稍候……")
resp = llm.invoke([HumanMessage(content=full_prompt)])

正在调用 qwen3.7-plus（已关闭 thinking），请稍候……


### 模型输出

In [9]:
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

### 1. 已给出或已验证的事实
*   **旅行日期**：2027年5月1日至5月5日（共5天4晚）。
*   **出行人数**：2人。
*   **目的地城市**：上海、苏州、杭州。
*   **必游景点**：西湖（位于杭州）。
*   **预算限制**：酒店每晚不超过800元。
*   **交通偏好**：尽量不要安排夜间跨城交通。
*   **未定事项**：城市游览顺序尚未确定。
*   **需求内容**：路线规划、酒店推荐、天气预测、美食攻略。

### 2. 需要查阅的事实
*   **2027年5月1日-5日的具体天气预报**：需查阅长期气象预测模型或历史同期气候数据（来源：中国气象局、AccuWeather等权威气象机构的历史数据或长期趋势报告，注意长期预报存在不确定性）。
*   **2027年五一假期的高速公路收费及拥堵情况预测**：需查阅交通运输部发布的节假日通行政策及往年同期交通大数据报告。
*   **上海、苏州、杭州三地间的高铁/火车时刻表及票价**：需查阅12306官网或相关铁路运营信息（注：具体时刻表通常在临近日期发布，目前可参考当前运行图作为基准，但需注明可能变动）。
*   **符合预算（≤800元/晚）且评价良好的酒店实时库存与价格**：需查阅携程、Booking.com、美团等OTA平台在2027年该时间段的预订情况（注：由于时间跨度大，目前无法获取确切实时价格，需基于当前价格水平进行预估或说明查询时效性）。
*   **各城市主要景点的开放时间及门票政策**：需查阅各景区官方网站或官方公众号发布的最新公告。

### 3. 需要推导的事实
*   **最优城市游览顺序**：基于地理位置（上海-苏州-杭州呈三角形分布，苏州与上海距离最近，杭州稍远）和交通便捷度，推导出一条减少折返、符合“不夜间跨城”约束的逻辑路线（例如：上海进->苏州->杭州出，或反之）。
*   **每日行程的时间分配可行性**：根据景点之间的距离、预计游览时长和交通耗时，推导每天是否能合理安排而不过于紧凑。
*   **总预算估算**：基于酒店上限（800元/晚 * 4晚 = 3200元）加上餐饮、交通和门票费用，推导两人旅行的总花费范围。
*   **天气对行程的影响评估**：根据历史气候数据推导5月初江南地区下雨的概率，从而建议是否携带雨具或安排室内备选方案。

### 4. 有根据的猜测
*   **人流高峰判断**：5月1日至5日为中国法定劳动节假期，推测这三个热门旅游城市将面临极高的人流密度，景点预约难度大，交通拥堵严重。
*   **酒店价格波动**：推测在五一黄金周期间，平时价格在400-500元的酒店可能会涨至800元甚至更高，因此“每晚不超过800元”的预算在核心区域可能较难找到高品质酒店，可能需要选择非核心区或提前很久预订。
*   **天气状况**：基于江南地区历史气候，5月初通常气温适宜（15-25摄氏度），但也是梅雨季节前夕，降雨概率较高，推测可能有阵雨或阴天。
*   **高铁票务紧张程度**：推测城际高铁票在假期期间会非常紧俏，尤其是上海到苏州、杭州方向的短途车次，可能需要抢票或接受候补。

## 5. 小结

- 本例的**思维链提示词**由固定说明、通过 `{task}` 嵌入的任务，以及四个指定的输出标题组成，并作为单条用户消息提交。
- 在 **LangChain** 中，`PromptTemplate` 负责模板渲染，`ChatOpenAI` 负责向模型服务发起调用。